# Experiment 2 — Staging on Wound-Only (Adaptively Dilated) — Colab Run

Trains a single config — **MiT-B1 + CORN, ImageNet weights** (task_3's confirmed best config) — on `datasets/wound-staging-masks-exp-2/`, using the original PIID 3-fold split (same filenames, same seed as the original Task 3 ablation). config_id = `mit_b1__corn__exp2_wound_dilated`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Upload `wound-staging-masks-exp-2.zip` to `drive/MyDrive/thesis/` first (built locally by `more_experiments/exp/exp2_wound_dilated/scripts/generate_dataset.py`, not re-derived here — no need to also upload the raw PIID zip or the leakage-free Task 1 checkpoints just to run this experiment).

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-staging-masks-exp-2.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-staging-masks-exp-2"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

Same `requirements.txt` as Task 3 -- this script reuses `task_3/src` directly rather than duplicating it.

In [ ]:
import os
!pip install -q -r {REPO_DIR}/task_3/requirements.txt
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp2_wound_dilated/scripts")
print(os.getcwd())

## 5. Redirect checkpoints/results to Drive

So a killed/disconnected runtime doesn't lose completed-fold history or checkpoints.

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp2_wound_dilated_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 6. Train -- MiT-B1 + CORN, 3-fold

Single config, no grid. `train.py` already skips completed `(config, fold)` rows and resumes a killed fold from its last saved epoch -- but a crash still kills the Python process itself, so this loop restarts it automatically. Output is streamed line-by-line (`Popen` + `-u`). A full runtime disconnect kills the loop too (Colab tears down the kernel); at that point just re-run this cell -- training picks up from the last saved epoch either way.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "train.py", "--workers", "4"]  # no --batch override: use configs/base.yaml's batch_size=16, matching task_3's own baseline ablation run exactly
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("training finished")
        break
    print(f"train.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 7. Check results

3 rows expected (one per fold) once training completes.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
print("mean test accuracy:", df["accuracy"].mean())
print("mean test macro_f1: ", df["macro_f1"].mean())
print("mean test qwk:      ", df["qwk"].mean())